# Solutions: Using configuration files in your RAP pipeline

This notebook provides step-by-step solutions for adding and using configuration parameters in a RAP pipeline. Each step matches the corresponding exercise notebook and demonstrates a modular approach.

In [ ]:
# Setup code for solutions

import pandas as pd
import yaml

# Load config
with open("../../config/user_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Load data
health_df = pd.read_csv("../../data/input/health_data.csv")

# output path
output_path = "../../data/outputs/cleaned/health_data_cleaned"

## Step 1 Solution: Add a new parameter to the config file

Open `config/user_config.yaml` and add the following parameter:

```yaml
min_height_cm: 120  # Minimum height (cm) to include in analysis
```

This parameter allows you to control which rows are included based on height. Save the file after editing.

## Step 2 Solution: Read config parameters in a modular way

Create a function to read config values in a new module, e.g., `src/python_rap_demo/io.py`:

In [ ]:
# src/main.py

min_height = config["min_height"]


In [ ]:
print(min_height)

## Step 3 Solution: Use config parameters in your pipeline

Update your pipeline (e.g., in `main.py` or a cleaning module) to use the config-driven parameter for filtering:

In [ ]:
# src/python_rap_demo/cleaning.py
import pandas as pd

def filter_by_min_height(df: pd.DataFrame, min_height_cm: int) -> pd.DataFrame:
    """
    Filter DataFrame to exclude rows below minimum height.

    Args:
        df (pd.DataFrame): Input health data.
        min_height_cm (int): Minimum height in cm.
    Returns:
        pd.DataFrame: Filtered DataFrame.
    """
    return df[df["height_cm"] >= min_height_cm]

Example usage in `main.py`
```python
from python_rap_demo.io import read_config
from python_rap_demo.cleaning import filter_by_min_height
```

In [ ]:
min_height_cm = config["min_height"]

filtered_df = filter_by_min_height(health_df, min_height_cm)

## Step 4 Solution: Add and use another config parameter (output format)

Add another parameter to your config file:

```yaml
output_format: csv  # Output file format: csv or xlsx
```

Create a modular function to save output in the desired format, e.g., in `src/python_rap_demo/io.py`:

In [ ]:
# src/python_rap_demo/io.py
import pandas as pd

def save_output(df: pd.DataFrame, output_path: str, output_format: str) -> None:
    """
    Save DataFrame to disk in the specified format.

    Args:
        df (pd.DataFrame): Data to save.
        output_path (str): Path (without extension) to save file.
        output_format (str): 'csv' or 'xlsx'.
    """
    if output_format == "csv":
        df.to_csv(f"{output_path}.csv", index=False)
    elif output_format == "xlsx":
        df.to_excel(f"{output_path}.xlsx", index=False)
    else:
        raise ValueError("Unsupported output format.")

In [ ]:
# Example usage in main.py
from python_rap_demo.io import save_output

output_format = config["output_format"]
save_output(filtered_df, output_path, output_format)